# 4. De una neurona a una red por capas

**Pregunta guía:** ¿qué ganamos al organizar varias neuronas?

![Red neuronal organizada en capas](imagenes/red_por_capas.png)

Una red agrupa unidades en tres tipos de capas:

1. La **entrada** contiene las características.
2. Las **capas ocultas** construyen representaciones intermedias.
3. La **salida** produce la respuesta final.

“Oculta” solo significa que no corresponde directamente a los datos observados ni a la respuesta.

In [1]:
import torch
import matplotlib.pyplot as plt
AMARILLO, NEGRO, BLANCO, GRIS = "#F9C80E", "#111111", "#FFFFFF", "#B7B7B7"
torch.set_printoptions(precision=3, sci_mode=False)
print(f"PyTorch {torch.__version__}")

PyTorch 2.14.0


## Una capa procesa varias neuronas a la vez

Reunimos los pesos en una matriz $W$ y los sesgos en un vector $b$:

$$h=a(XW+b).$$

Cada columna de $W$ contiene los pesos de una neurona. El número de columnas determina cuántas salidas produce la capa.

In [2]:
X = torch.tensor([[0.2,0.8,0.4], [0.9,0.1,0.6]])
W = torch.tensor([[1.,-1.,.5,.2], [.5,1.,-.5,.8], [-.2,.4,1.,-1.]])
b = torch.tensor([-.4,-.2,-.3,.1])
z = X @ W + b
h = (z >= 0).to(torch.int)
print("entrada:", X.shape)
print("pesos:  ", W.shape)
print("salida: ", h.shape)
print("\nActivaciones:\n", h)

entrada: torch.Size([2, 3])
pesos:   torch.Size([3, 4])
salida:  torch.Size([2, 4])

Activaciones:
 tensor([[1, 1, 0, 1],
        [1, 0, 1, 0]], dtype=torch.int32)


## Composición de decisiones

Varias neuronas detectan condiciones parciales; otra puede combinarlas. Esto crea regiones que una sola recta no describe.

El ejemplo implementa XOR con parámetros elegidos a mano:

- Una unidad oculta calcula OR.
- Otra calcula NAND.
- La salida calcula AND sobre ambas señales.

Estudiamos el recorrido de la información, no cómo se obtienen automáticamente los parámetros.

In [3]:
X_logico = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
umbral = lambda z: (z >= 0).to(torch.float32)
or_oculta = umbral(X_logico @ torch.tensor([1.,1.]) - .5)
nand_oculta = umbral(X_logico @ torch.tensor([-1.,-1.]) + 1.5)
H = torch.stack([or_oculta, nand_oculta], dim=1)
xor_salida = umbral(H @ torch.tensor([1.,1.]) - 1.5)
print("x₁ x₂ | OR NAND | XOR")
for x, h, y in zip(X_logico, H, xor_salida):
    print(f" {int(x[0])}  {int(x[1])} |  {int(h[0])}    {int(h[1])}  |  {int(y)}")

x₁ x₂ | OR NAND | XOR
 0  0 |  0    1  |  0
 0  1 |  1    1  |  1
 1  0 |  1    1  |  1
 1  1 |  1    0  |  0


## Anchura, profundidad y propagación hacia delante

La **anchura** es el número de neuronas de una capa. La **profundidad** cuenta transformaciones sucesivas. Una red con capa oculta suele llamarse perceptrón multicapa.

Aquí solo necesitamos la propagación **hacia delante**:

$$x\longrightarrow h\longrightarrow y.$$

Cada capa recibe la salida anterior. El ajuste automático de parámetros pertenece a una etapa posterior.

## Comprueba tu comprensión

1. Identifica las formas de $X$, $W$, $b$ y $h$.
2. ¿Por qué cada columna de $W$ representa una neurona?
3. Sigue manualmente $(1,0)$ a través de OR, NAND y AND.
4. Distingue entrada, capa oculta y salida.
5. Explica qué aporta la composición frente a una frontera.

**Cierre:** un perceptrón convierte características en una decisión lineal; una red organiza perceptrones en capas para componer decisiones.